In [3]:

# Import required libraries
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# Load the Breast Cancer dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Dataset loaded successfully!")
print("Total rows:", X.shape[0])
print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


Dataset loaded successfully!
Total rows: 569
Training rows: 455
Testing rows: 114


In [5]:

# Grid Search hyperparameter tuning
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define the model
model = DecisionTreeClassifier(random_state=42)

# Specify parameter combinations to test
param_grid = {
    "max_depth": [2, 3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}

# Search all combinations using 5-fold cross-validation
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

# Evaluate the best model on test data
grid_predictions = grid_search.best_estimator_.predict(X_test)
grid_accuracy = accuracy_score(y_test, grid_predictions)

print("GRID SEARCH RESULTS")
print("Best parameters:", grid_search.best_params_)
print("Best cross-validation accuracy:", round(grid_search.best_score_, 4))
print("Test accuracy:", round(grid_accuracy, 4))


GRID SEARCH RESULTS
Best parameters: {'criterion': 'gini', 'max_depth': 5, 'min_samples_split': 5}
Best cross-validation accuracy: 0.9385
Test accuracy: 0.9211


In [7]:

# Random Search hyperparameter tuning
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "max_depth": [2, 3, 5, 10, 15, 20, None],
    "min_samples_split": [2, 3, 5, 10, 15],
    "criterion": ["gini", "entropy"]
}

random_search = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

random_predictions = random_search.best_estimator_.predict(X_test)
random_accuracy = accuracy_score(y_test, random_predictions)

print("RANDOM SEARCH RESULTS")
print("Best parameters:", random_search.best_params_)
print("Best cross-validation accuracy:", round(random_search.best_score_, 4))
print("Test accuracy:", round(random_accuracy, 4))


RANDOM SEARCH RESULTS
Best parameters: {'min_samples_split': 5, 'max_depth': 5, 'criterion': 'gini'}
Best cross-validation accuracy: 0.9385
Test accuracy: 0.9211


In [9]:

# Compare Grid Search and Random Search
comparison = pd.DataFrame({
    "Method": ["GridSearchCV", "RandomizedSearchCV"],
    "Best CV Accuracy": [
        grid_search.best_score_,
        random_search.best_score_
    ],
    "Test Accuracy": [
        grid_accuracy,
        random_accuracy
    ]
})

print(comparison.to_string(index=False))

# Identify the method with higher test accuracy
if grid_accuracy > random_accuracy:
    print("\nGrid Search achieved higher test accuracy.")
elif random_accuracy > grid_accuracy:
    print("\nRandom Search achieved higher test accuracy.")
else:
    print("\nBoth methods achieved the same test accuracy.")


            Method  Best CV Accuracy  Test Accuracy
      GridSearchCV          0.938462       0.921053
RandomizedSearchCV          0.938462       0.921053

Both methods achieved the same test accuracy.


In [11]:

import joblib

# Save the comparison results
comparison.to_csv("tuning_comparison.csv", index=False)

# Save the best models
joblib.dump(grid_search.best_estimator_, "grid_best_model.pkl")
joblib.dump(random_search.best_estimator_, "random_best_model.pkl")

print("Files saved successfully:")
print("1. tuning_comparison.csv")
print("2. grid_best_model.pkl")
print("3. random_best_model.pkl")


Files saved successfully:
1. tuning_comparison.csv
2. grid_best_model.pkl
3. random_best_model.pkl


In [12]:

from google.colab import files

files.download("tuning_comparison.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>